<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M10/M10_Lab1_MCP_Servers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🔌 M10 Lab 1 — MCP Servers</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand the <b>MCP</b> (Model Context Protocol) architecture</li>
    <li>Learn the roles of <b>clients</b>, <b>servers</b>, <b>tools</b>, and <b>resources</b></li>
    <li>Build a simulated <b>MCP tool server</b> with tool schemas</li>
    <li>Connect an LLM that <b>discovers and calls tools</b> dynamically</li>
    <li>Understand how MCP standardizes <b>AI-tool integration</b></li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

client = setup_openai()

import json

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ What Is MCP?</h2>
</div>

**MCP** (Model Context Protocol) is an open standard created by Anthropic that defines how AI models connect to external tools and data sources.

Think of MCP like **USB for AI** — before USB, every device needed its own cable. Before MCP, every tool integration required custom code.

### MCP Architecture

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   LLM Host  │────▶│  MCP Client │────▶│  MCP Server │
│  (Claude,   │     │  (in the    │     │  (exposes   │
│   GPT, etc) │     │   host app) │     │   tools)    │
└─────────────┘     └─────────────┘     └─────────────┘
                                              │
                                    ┌─────────┼─────────┐
                                    ▼         ▼         ▼
                                 📊 DB    📁 Files   🌐 APIs
```

| Component | Role | Example |
|-----------|------|---------|
| **Host** | The AI application | Claude Desktop, VS Code |
| **Client** | Manages connections to servers | Built into the host |
| **Server** | Exposes tools/resources | Gmail MCP, Database MCP |
| **Tools** | Actions the AI can perform | `send_email`, `query_db` |
| **Resources** | Data the AI can read | Files, database schemas |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Build an MCP-Style Tool Server</h2>
</div>

Let's simulate an MCP server that exposes three tools: a database query tool, a file reader, and a calculator. In production, this would use the actual MCP protocol over JSON-RPC — here we simulate the core pattern.

In [ ]:
# ============================================================
# 🔌 MCP-Style Tool Server
# ============================================================

class MCPToolServer:
    """A simulated MCP tool server that exposes tools with schemas."""

    def __init__(self, name: str):
        self.name = name
        self.tools = {}  # name → {schema, handler}

    def register_tool(self, name: str, description: str, parameters: dict, handler):
        """Register a tool with its schema and handler function."""
        self.tools[name] = {
            "schema": {
                "type": "function",
                "function": {
                    "name": name,
                    "description": description,
                    "parameters": parameters
                }
            },
            "handler": handler
        }
        print(f"  ✅ Registered tool: {name}")

    def list_tools(self) -> list:
        """Return all tool schemas (MCP tools/list equivalent)."""
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name: str, arguments: dict) -> str:
        """Execute a tool by name (MCP tools/call equivalent)."""
        if name not in self.tools:
            return json.dumps({"error": f"Unknown tool: {name}"})
        try:
            result = self.tools[name]["handler"](arguments)
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})

print("✅ MCPToolServer class defined")

In [ ]:
# ============================================================
# 📊 Register Tools: Database, File Reader, Calculator
# ============================================================

# Create the server
server = MCPToolServer("data-tools")
print(f"🔌 MCP Server: '{server.name}'\n")

# --- Tool 1: Database Query ---
# Mock database
MOCK_DB = {
    "employees": [
        {"id": 1, "name": "Alice", "dept": "Engineering", "salary": 120000},
        {"id": 2, "name": "Bob", "dept": "Marketing", "salary": 95000},
        {"id": 3, "name": "Carol", "dept": "Engineering", "salary": 135000},
        {"id": 4, "name": "Dave", "dept": "Sales", "salary": 105000},
        {"id": 5, "name": "Eve", "dept": "Engineering", "salary": 128000},
    ],
    "products": [
        {"id": 1, "name": "Widget A", "price": 29.99, "stock": 150},
        {"id": 2, "name": "Widget B", "price": 49.99, "stock": 75},
        {"id": 3, "name": "Widget C", "price": 19.99, "stock": 300},
    ]
}

def query_database(args):
    table = args["table"]
    if table not in MOCK_DB:
        return f"Table '{table}' not found. Available: {list(MOCK_DB.keys())}"
    rows = MOCK_DB[table]
    # Optional filter
    if "filter_field" in args and "filter_value" in args:
        rows = [r for r in rows if str(r.get(args["filter_field"])) == str(args["filter_value"])]
    return {"table": table, "count": len(rows), "rows": rows}

server.register_tool(
    "query_database",
    "Query a database table. Optionally filter by a field value. Available tables: employees, products.",
    {
        "type": "object",
        "properties": {
            "table": {"type": "string", "description": "Table name: employees or products"},
            "filter_field": {"type": "string", "description": "Optional: field to filter by"},
            "filter_value": {"type": "string", "description": "Optional: value to match"}
        },
        "required": ["table"]
    },
    query_database
)

# --- Tool 2: File Reader ---
MOCK_FILES = {
    "report.txt": "Q4 2024 Revenue: $2.3M. Growth rate: 15%. Key driver: Enterprise contracts.",
    "config.json": '{"model": "gpt-4.1-mini", "temperature": 0.7, "max_tokens": 1000}',
    "notes.md": "# Meeting Notes\n- Review Q4 results\n- Plan Q1 roadmap\n- Hire 3 engineers",
}

def read_file(args):
    filename = args["filename"]
    if filename not in MOCK_FILES:
        return f"File '{filename}' not found. Available: {list(MOCK_FILES.keys())}"
    return {"filename": filename, "content": MOCK_FILES[filename]}

server.register_tool(
    "read_file",
    "Read the contents of a file. Available files: report.txt, config.json, notes.md.",
    {
        "type": "object",
        "properties": {
            "filename": {"type": "string", "description": "Name of the file to read"}
        },
        "required": ["filename"]
    },
    read_file
)

# --- Tool 3: Calculator ---
def calculate(args):
    expr = args["expression"]
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expr):
        return "Error: Invalid characters"
    return {"expression": expr, "result": str(round(eval(expr), 4))}

server.register_tool(
    "calculator",
    "Evaluate a mathematical expression. Supports basic arithmetic.",
    {
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Math expression, e.g. '120000 * 3'"}
        },
        "required": ["expression"]
    },
    calculate
)

print(f"\n📋 Server has {len(server.tools)} tools registered")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> In real MCP, the server advertises its tools over JSON-RPC, and the client discovers them at runtime. Our simulation follows the same pattern — the LLM will discover tools from <code>list_tools()</code> and call them via <code>call_tool()</code>.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Connect the LLM to the Server</h2>
</div>

Now we connect an LLM that discovers tools from the server and uses them to answer questions — just like a real MCP client would.

In [ ]:
# ============================================================
# 🤖 MCP Client: LLM Discovers and Calls Tools
# ============================================================

def mcp_agent(question: str, server: MCPToolServer, verbose: bool = True) -> str:
    """An LLM-powered agent that uses MCP server tools."""

    # Step 1: Discover tools from the server
    tool_schemas = server.list_tools()
    if verbose:
        print(f"\n{'='*70}")
        print(f"🧑 Question: {question}")
        print(f"🔌 Discovered {len(tool_schemas)} tools from server '{server.name}'")
        print(f"{'='*70}")

    # Step 2: Send question + tool schemas to LLM
    messages = [
        {"role": "system", "content": "You have access to tools provided by an MCP server. Use them to answer questions accurately."},
        {"role": "user", "content": question}
    ]

    for step in range(1, 6):
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tool_schemas,
            temperature=0
        )

        message = response.choices[0].message

        if message.tool_calls:
            messages.append(message)
            for tc in message.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"\n  🔵 Step {step}: {fn_name}({json.dumps(fn_args, indent=2)[:100]})")

                # Call the MCP server
                result = server.call_tool(fn_name, fn_args)
                if verbose:
                    print(f"  👁️ Result: {result[:150]}..." if len(result) > 150 else f"  👁️ Result: {result}")

                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            if verbose:
                print(f"\n  🟢 Final Answer:")
                print(f"  {message.content}")
                print(f"{'='*70}")
            return message.content

    return "Agent exceeded max steps."

print("✅ MCP agent ready!")

In [ ]:
# ============================================================
# 🧪 Test 1: Database Query
# ============================================================
mcp_agent("How many employees are in the Engineering department and what is their average salary?", server)

In [ ]:
# ============================================================
# 🧪 Test 2: File Reading
# ============================================================
mcp_agent("What was the Q4 revenue according to the report file?", server)

In [ ]:
# ============================================================
# 🧪 Test 3: Multi-Tool Question
# ============================================================
mcp_agent(
    "What is the total salary cost for all employees? Also, read the meeting notes and tell me what's planned.",
    server
)

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> MCP standardizes how AI models connect to external tools. Instead of building custom integrations for each tool, you build one MCP server per tool — and <b>any MCP-compatible AI host</b> can use it. This is why MCP is called "USB for AI."
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the best analogy for MCP (Model Context Protocol)?",
        "options": [
            "A programming language for AI models",
            "USB for AI — a standard interface between AI models and external tools",
            "A database management system for LLMs",
            "A training framework for fine-tuning models"
        ],
        "answer": 1,
        "explanation": "Just like USB standardized how devices connect to computers, MCP standardizes how AI models connect to external tools and data sources. Build one MCP server, and any MCP-compatible host can use it."
    },
    {
        "q": "In MCP architecture, what does the 'server' do?",
        "options": [
            "It runs the LLM model and generates responses",
            "It manages user authentication and sessions",
            "It exposes tools and resources that AI models can discover and use",
            "It stores the conversation history in a database"
        ],
        "answer": 2,
        "explanation": "The MCP server exposes tools (actions the AI can perform) and resources (data the AI can read). The server advertises its capabilities, and the client (in the AI host) discovers and calls them."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a New Tool to the Server

Add a `web_search` tool to the MCP server that simulates web search results. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Add a Web Search Tool
# ============================================================
# Replace each ----- with the correct value

# Mock search results
SEARCH_DB = {
    "mcp protocol": "MCP (Model Context Protocol) is an open standard by Anthropic for connecting AI to tools.",
    "langchain": "LangChain is a framework for building LLM-powered applications with chains and agents.",
    "rag": "RAG (Retrieval-Augmented Generation) combines search with LLM generation for factual answers.",
}

def web_search(args):
    query = args["query"].lower()
    for key, value in SEARCH_DB.items():
        if key in query:
            return {"query": args["query"], "result": value}
    return {"query": args["query"], "result": "No results found."}

server.register_tool(
    "-----",                              # What name for this tool?
    "Search the web for information about AI topics.",
    {
        "type": "-----",                  # What JSON Schema type?
        "properties": {
            "query": {
                "type": "string",
                "description": "-----"    # Describe what this parameter does
            }
        },
        "required": ["query"]
    },
    -----                                  # What handler function?
)

# Test it!
mcp_agent("What is MCP protocol and how does it relate to LangChain?", server)

**📝 Your Observations:** *(double-click to edit this cell)*

1. How does the LLM decide which tool to call? Does it always choose correctly? _[Your answer]_

2. What would happen if two tools had similar descriptions? _[Your answer]_

3. How is MCP different from the function calling we did in M03? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>MCP Resources:</b> Add a <code>list_resources()</code> method that returns available data sources (like database schemas)</li>
    <li><b>Multi-server:</b> Create a second MCPToolServer (e.g., "email-tools") and connect both to the agent</li>
    <li><b>Real MCP:</b> Explore the <a href="https://modelcontextprotocol.io/">MCP documentation</a> and try building a real server with the Python SDK</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built an MCP-style tool server and connected it to an LLM agent.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>MCP</b> = standard protocol for AI-tool integration ("USB for AI")</li>
      <li><b>Servers</b> expose tools with schemas; <b>clients</b> discover and call them</li>
      <li>Tools have <b>name + description + parameter schema</b> — the LLM reads these to decide what to call</li>
      <li>MCP enables <b>composable AI</b> — mix and match servers for any use case</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M10 Lab 2: Guardrails & Safety</p>
</div>